In [37]:
!pip install wandb
!pip install webrtcvad

In [38]:
import numpy as np
import pandas as pd
import os
import random
import glob
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, accuracy_score
from kaggle_secrets import UserSecretsClient
import wandb
import shutil
import webrtcvad
import struct

# Set random seeds for reproducibility
SEED = 42

# --- Device Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Paths and Spectrogram Configuration ---
REAL_AUDIO_PATH = "/kaggle/input/the-lj-speech-dataset/LJSpeech-1.1/wavs"
FAKE_AUDIO_PATH_PARENT = "/kaggle/input/wavefake-test/generated_audio"

MANUAL_TEST_EXPORT_DIR = "/kaggle/working/manual_dataset_export" 
os.makedirs(os.path.join(MANUAL_TEST_EXPORT_DIR, "real"), exist_ok=True)
os.makedirs(os.path.join(MANUAL_TEST_EXPORT_DIR, "fake"), exist_ok=True)

SR = 16000
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
MAX_FRAMES_SPEC = 313 
FMIN = 0.0
FMAX = None
APPLY_AUGMENTATION = True 
NUM_TIME_MASKS = 1
NUM_FREQ_MASKS = 1
TIME_MASK_MAX_WIDTH = 40
FREQ_MASK_MAX_WIDTH = 15
NORM_EPSILON = 1e-6
MASK_REPLACEMENT_VALUE = 0.0
LIMIT_FILES = None 
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

# --- Training Hyperparameters (Common) ---
BATCH_SIZE = 32
NUM_WORKERS = 2 
LEARNING_RATE = 1e-4
EPOCHS = 20 
WEIGHT_DECAY = 1e-4
PATIENCE_LIMIT = 7 

# --- Model-Specific Base Hyperparameters ---
VIT_PATCH_SIZE = 16
VIT_BASE_DROP_RATE = 0.1
VIT_BASE_ATTN_DROP_RATE = 0.1
CNN_BASE_DROPOUT_RATE = 0.4

# --- VAD Parameters ---
VAD_AGGRESSIVENESS = 1       
VAD_FRAME_DURATION_MS = 30 
VAD_MIN_SPEECH_DURATION_MS = 200
TARGET_AUDIO_DURATION_SEC_AFTER_VAD = 10
VAD_KEEP_ONLY_FIRST_SEGMENT = False


# --- WandB Login with Kaggle Secrets ---
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api_key")
    wandb.login(key=wandb_api_key)
    print("WandB login successful using Kaggle Secrets.")
except Exception as e:
    print(f"Failed to login to WandB via Kaggle Secrets: {e}. Falling back to environment variable or manual login.")
    try:
        wandb.login()
        print("WandB login successful (manual/env var).")
    except Exception as e2:
        print(f"Also failed to login manually: {e2}")

Using device: cuda


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


WandB login successful using Kaggle Secrets.


In [39]:
# --- 1. Data Loading and Preprocessing Functions ---
def get_and_split_audio_files_with_manual_test(
    real_dir,
    fake_dir_parent,
    fake_ljspeech_keyword_for_main_train=None,
    target_total_fake_for_main_train=None,
    num_manual_test_samples_per_class=100,
    keyword_for_manual_test_fake=None, 
    manual_test_export_path=None
):
    print("--- Starting Data Preparation: Separating Manual Test Set & Preparing Main Train/Val/Auto-Test Data ---")
    all_source_real_files = glob.glob(os.path.join(real_dir, '*.wav'))
    if len(all_source_real_files) < num_manual_test_samples_per_class:
        raise ValueError(f"Not enough Real files ({len(all_source_real_files)}) for manual test set ({num_manual_test_samples_per_class}).")
    random.shuffle(all_source_real_files) 
    manual_test_real_files = all_source_real_files[:num_manual_test_samples_per_class]
    remaining_real_files_for_main_process = all_source_real_files[num_manual_test_samples_per_class:]
    print(f"Selected {len(manual_test_real_files)} Real files for manual test.")
    print(f"{len(remaining_real_files_for_main_process)} Real files remaining for main process.")

    all_source_fake_files_pool = []
    if os.path.exists(fake_dir_parent) and os.path.isdir(fake_dir_parent):
        fake_subdirs_all = [os.path.join(fake_dir_parent, d) for d in os.listdir(fake_dir_parent)
                            if os.path.isdir(os.path.join(fake_dir_parent, d))]
        for subdir in fake_subdirs_all:
            all_source_fake_files_pool.extend(glob.glob(os.path.join(subdir, '*.wav')))
    else:
        print(f"Warning: Fake parent directory '{fake_dir_parent}' does not exist or is not a directory.")

    if not all_source_fake_files_pool:
        print("Warning: No fake files found in any subdirectories of WaveFake. Manual test fake and main fake will be empty.")

    manual_test_fake_files = []
    temp_fake_pool_for_manual_selection = list(all_source_fake_files_pool) 
    random.shuffle(temp_fake_pool_for_manual_selection)

    if keyword_for_manual_test_fake:
        keyword_matched_fakes = [
            f for f in temp_fake_pool_for_manual_selection 
            if keyword_for_manual_test_fake.lower() in os.path.basename(os.path.dirname(f)).lower() or \
               keyword_for_manual_test_fake.lower() in os.path.basename(f).lower()
        ]
        other_fakes_for_manual = [f for f in temp_fake_pool_for_manual_selection if f not in keyword_matched_fakes]
        
        if len(keyword_matched_fakes) >= num_manual_test_samples_per_class:
            manual_test_fake_files = random.sample(keyword_matched_fakes, num_manual_test_samples_per_class)
        else:
            manual_test_fake_files.extend(keyword_matched_fakes)
            needed_more = num_manual_test_samples_per_class - len(manual_test_fake_files)
            if needed_more > 0 and other_fakes_for_manual:
                if len(other_fakes_for_manual) >= needed_more:
                    manual_test_fake_files.extend(random.sample(other_fakes_for_manual, needed_more))
                else:
                    manual_test_fake_files.extend(other_fakes_for_manual)
        print(f"Selected {len(manual_test_fake_files)} Fake files for manual test (prioritizing keyword: '{keyword_for_manual_test_fake}').")
    else: 
        if len(temp_fake_pool_for_manual_selection) >= num_manual_test_samples_per_class:
            manual_test_fake_files = random.sample(temp_fake_pool_for_manual_selection, num_manual_test_samples_per_class)
            print(f"Selected {len(manual_test_fake_files)} Fake files randomly for manual test.")
        else:
            manual_test_fake_files = list(temp_fake_pool_for_manual_selection)
            print(f"Warning: Not enough fake files ({len(manual_test_fake_files)}) for manual test. Using all available.")

    if manual_test_export_path:
        dest_manual_real = os.path.join(manual_test_export_path, "real")
        dest_manual_fake = os.path.join(manual_test_export_path, "fake")
        os.makedirs(dest_manual_real, exist_ok=True)
        os.makedirs(dest_manual_fake, exist_ok=True)
        for f_path in manual_test_real_files:
            try: shutil.copy(f_path, os.path.join(dest_manual_real, os.path.basename(f_path)))
            except Exception as e: print(f"Failed to copy {f_path} to manual real: {e}")
        for f_path in manual_test_fake_files:
            try: shutil.copy(f_path, os.path.join(dest_manual_fake, os.path.basename(f_path)))
            except Exception as e: print(f"Failed to copy {f_path} to manual fake: {e}")
        print(f"Exported manual test files to {manual_test_export_path}")

    final_manual_test_filepaths = manual_test_real_files + manual_test_fake_files
    final_manual_test_labels = [0] * len(manual_test_real_files) + [1] * len(manual_test_fake_files)
    
    combined_manual_test = list(zip(final_manual_test_filepaths, final_manual_test_labels))
    random.shuffle(combined_manual_test)
    final_manual_test_filepaths_shuffled, final_manual_test_labels_shuffled = [], []
    if combined_manual_test:
        final_manual_test_filepaths_shuffled, final_manual_test_labels_shuffled = zip(*combined_manual_test)
    
    print(f"Manual test set created with {len(final_manual_test_filepaths_shuffled)} files "
          f"(Real: {final_manual_test_labels_shuffled.count(0)}, Fake: {final_manual_test_labels_shuffled.count(1)}).")

    remaining_fake_files_pool_for_main = [
        f for f in all_source_fake_files_pool if f not in manual_test_fake_files
    ]
    
    if target_total_fake_for_main_train is None:
        target_total_fake_for_main_train = len(remaining_real_files_for_main_process)
        print(f"Targeting {target_total_fake_for_main_train} fake files for main process.")

    train_val_ljspeech_fake_files_pool = []
    if fake_ljspeech_keyword_for_main_train: 
        for f_path in remaining_fake_files_pool_for_main:
            parent_dir_name = os.path.basename(os.path.dirname(f_path))
            if fake_ljspeech_keyword_for_main_train.lower() in parent_dir_name.lower() or \
               fake_ljspeech_keyword_for_main_train.lower() in os.path.basename(f_path).lower() :
                 train_val_ljspeech_fake_files_pool.append(f_path)
        print(f"Found {len(train_val_ljspeech_fake_files_pool)} LJSpeech-related fake files in remaining pool for main process.")
    else: 
        train_val_ljspeech_fake_files_pool = list(remaining_fake_files_pool_for_main)
        print(f"Using all {len(train_val_ljspeech_fake_files_pool)} remaining fake files for main process (no LJSpeech keyword filter).")
            
    selected_fake_for_main_process = []
    if not train_val_ljspeech_fake_files_pool:
        print(f"Warning: No fake files matching criteria found for main process.")
    elif len(train_val_ljspeech_fake_files_pool) <= target_total_fake_for_main_train:
        selected_fake_for_main_process = list(train_val_ljspeech_fake_files_pool)
        print(f"Using all {len(selected_fake_for_main_process)} available filtered fake files for main process.")
    else:
        selected_fake_for_main_process = random.sample(train_val_ljspeech_fake_files_pool, target_total_fake_for_main_train)
        print(f"Randomly sampled {len(selected_fake_for_main_process)} filtered fake files for main process from "
              f"{len(train_val_ljspeech_fake_files_pool)} available.")

    final_main_process_filepaths = remaining_real_files_for_main_process + selected_fake_for_main_process
    final_main_process_labels = [0] * len(remaining_real_files_for_main_process) + [1] * len(selected_fake_for_main_process)

    combined_main_process = list(zip(final_main_process_filepaths, final_main_process_labels))
    random.shuffle(combined_main_process)
    final_main_process_filepaths_shuffled, final_main_process_labels_shuffled = [], []
    if combined_main_process:
        final_main_process_filepaths_shuffled, final_main_process_labels_shuffled = zip(*combined_main_process)

    print(f"Total files for main process (train/val/auto-test): {len(final_main_process_filepaths_shuffled)} "
          f"(Real: {final_main_process_labels_shuffled.count(0)}, Fake: {final_main_process_labels_shuffled.count(1)}).")
    print("--- Data Preparation Finished ---")
    
    return (list(final_manual_test_filepaths_shuffled), list(final_manual_test_labels_shuffled),
            list(final_main_process_filepaths_shuffled), list(final_main_process_labels_shuffled))

def audio_to_melspectrogram(
    filepath, 
    sr=SR, 
    n_fft=N_FFT, 
    hop_length=HOP_LENGTH, 
    n_mels=N_MELS, 
    max_frames_spec=MAX_FRAMES_SPEC,
    fmin=FMIN, 
    fmax=FMAX,
    use_vad=True,
    vad_aggressiveness=VAD_AGGRESSIVENESS,
    vad_frame_duration_ms=VAD_FRAME_DURATION_MS,
    min_speech_duration_ms=VAD_MIN_SPEECH_DURATION_MS,
    target_audio_duration_sec=TARGET_AUDIO_DURATION_SEC_AFTER_VAD,
    keep_only_first_speech_segment=VAD_KEEP_ONLY_FIRST_SEGMENT 
):
    try:
        y, sr_orig = librosa.load(filepath, sr=None)
        
        if y.ndim > 1: 
            y = librosa.to_mono(y.T if y.shape[0] > y.shape[1] else y)

        if sr_orig != sr:
            y = librosa.resample(y, orig_sr=sr_orig, target_sr=sr)

        y_processed = y

        if use_vad:
            if y.dtype != np.int16:
                y_int16 = (y * 32767).astype(np.int16)
            else:
                y_int16 = y

            vad = webrtcvad.Vad()
            vad.set_mode(vad_aggressiveness)
            samples_per_frame = int(sr * vad_frame_duration_ms / 1000)
            
            speech_segments_audio = []
            current_segment_frames = []
            triggered = False

            for i in range(0, len(y_int16) - samples_per_frame + 1, samples_per_frame):
                frame = y_int16[i : i + samples_per_frame]
                if len(frame) < samples_per_frame: continue

                try:
                    frame_bytes = struct.pack(f"{len(frame)}h", *frame)
                    is_speech = vad.is_speech(frame_bytes, sr)
                except Exception: 
                    is_speech = False 

                if is_speech:
                    current_segment_frames.append(frame)
                    triggered = True
                elif triggered:
                    if current_segment_frames:
                        full_segment_audio = np.concatenate(current_segment_frames)
                        if len(full_segment_audio) >= int(sr * min_speech_duration_ms / 1000):
                            speech_segments_audio.append(full_segment_audio)
                            if keep_only_first_speech_segment:
                                break 
                    current_segment_frames = []
                    triggered = False
            
            if triggered and current_segment_frames:
                full_segment_audio = np.concatenate(current_segment_frames)
                if len(full_segment_audio) >= int(sr * min_speech_duration_ms / 1000):
                    speech_segments_audio.append(full_segment_audio)

            if not speech_segments_audio: return None 

            if keep_only_first_speech_segment and speech_segments_audio:
                y_processed_int16 = speech_segments_audio[0]
            elif speech_segments_audio:
                y_processed_int16 = np.concatenate(speech_segments_audio)
            else: return None
            
            y_processed = y_processed_int16.astype(np.float32) / 32767.0
        
        # Xử lý target_audio_duration_sec sau VAD (hoặc sau khi load nếu không dùng VAD)
        if target_audio_duration_sec is not None:
            target_samples = int(target_audio_duration_sec * sr)
            if len(y_processed) > target_samples:
                if len(y_processed) - target_samples > 0:
                    start_idx = random.randint(0, len(y_processed) - target_samples)
                    y_processed = y_processed[start_idx : start_idx + target_samples]
            elif len(y_processed) < target_samples and len(y_processed) > 0: 
                padding = np.zeros(target_samples - len(y_processed), dtype=np.float32)
                y_processed = np.concatenate((y_processed, padding))
        
        if len(y_processed) < hop_length: return None

        mel_spectrogram = librosa.feature.melspectrogram(
            y=y_processed, sr=sr, n_fft=n_fft, hop_length=hop_length, 
            n_mels=n_mels, fmin=fmin, fmax=fmax if fmax is not None else sr/2
        )
        log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
        
        current_frames = log_mel_spectrogram.shape[1]
        if current_frames == 0: return None
            
        if current_frames < max_frames_spec:
            pad_value = log_mel_spectrogram.min() 
            pad_width = max_frames_spec - current_frames
            return np.pad(log_mel_spectrogram, ((0, 0), (0, pad_width)), mode='constant', constant_values=pad_value)
        elif current_frames > max_frames_spec:
            start_frame = (current_frames - max_frames_spec) // 2
            return log_mel_spectrogram[:, start_frame : start_frame + max_frames_spec]
        else: 
            return log_mel_spectrogram
            
    except Exception as e:
        return None

In [40]:
# --- 2. PyTorch Dataset ---
class AudioDataset(Dataset):
    def __init__(self, filepaths, labels, transform_spectrogram_fn, augment=False, is_vit_input=False,
                 time_mask_max_width=TIME_MASK_MAX_WIDTH, freq_mask_max_width=FREQ_MASK_MAX_WIDTH,
                 num_time_masks=NUM_TIME_MASKS, num_freq_masks=NUM_FREQ_MASKS,
                 mask_replacement_value=MASK_REPLACEMENT_VALUE,
                 use_vad_in_dataset=True, 
                 vad_params=None
                ):
        self.filepaths = filepaths
        self.labels = labels
        self.transform_spectrogram_fn = transform_spectrogram_fn 
        self.augment = augment
        self.is_vit_input = is_vit_input 
        self.time_mask_max_width = time_mask_max_width
        self.freq_mask_max_width = freq_mask_max_width
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks
        self.mask_replacement_value = mask_replacement_value
        self.use_vad_in_dataset = use_vad_in_dataset
        self.vad_params = vad_params if vad_params is not None else {}


    def __len__(self):
        return len(self.filepaths)

    def _apply_time_mask(self, spectrogram):
        augmented_spec = np.copy(spectrogram)
        num_frames = augmented_spec.shape[1]
        for _ in range(self.num_time_masks):
            if self.time_mask_max_width > 0 and num_frames > self.time_mask_max_width:
                t = random.randint(1, self.time_mask_max_width)
                t0 = random.randint(0, num_frames - t)
                augmented_spec[:, t0:t0 + t] = self.mask_replacement_value
        return augmented_spec


    def _apply_freq_mask(self, spectrogram):
        augmented_spec = np.copy(spectrogram)
        num_mels = augmented_spec.shape[0]
        for _ in range(self.num_freq_masks):
            if self.freq_mask_max_width > 0 and num_mels > self.freq_mask_max_width:
                f = random.randint(1, self.freq_mask_max_width)
                f0 = random.randint(0, num_mels - f)
                augmented_spec[f0:f0 + f, :] = self.mask_replacement_value
        return augmented_spec

    def __getitem__(self, idx):
        filepath = self.filepaths[idx]
        label = self.labels[idx]
        mel_spec = self.transform_spectrogram_fn(
            filepath, 
            use_vad=self.use_vad_in_dataset,
            **self.vad_params 
        )
        
        if mel_spec is None: return None 

        if self.augment: # Augmentation trên spectrogram
            mel_spec = self._apply_time_mask(mel_spec)
            mel_spec = self._apply_freq_mask(mel_spec)

        mean = np.mean(mel_spec)
        std = np.std(mel_spec)
        mel_spec_normalized = (mel_spec - mean) / (std + NORM_EPSILON)

        if self.is_vit_input:
            mel_spec_final = np.stack([mel_spec_normalized]*3, axis=0)
        else:
            mel_spec_final = np.expand_dims(mel_spec_normalized, axis=0)

        mel_spec_tensor = torch.tensor(mel_spec_final, dtype=torch.float32)
        label_tensor = torch.tensor(label, dtype=torch.float32) 
        return mel_spec_tensor, label_tensor

# collate_fn_skip_none_vit và collate_fn_skip_none_cnn giữ nguyên

In [41]:
# --- 3. Data Splitting ---
NUM_MANUAL_TEST_SAMPLES = 100
try:
    num_total_real_files_initial = len(glob.glob(os.path.join(REAL_AUDIO_PATH, '*.wav')))
    TARGET_FAKE_FOR_MAIN_TRAIN_VAL = num_total_real_files_initial - NUM_MANUAL_TEST_SAMPLES
    if TARGET_FAKE_FOR_MAIN_TRAIN_VAL < 0: TARGET_FAKE_FOR_MAIN_TRAIN_VAL = 0
except Exception as e:
    print(f"Error getting initial real file count: {e}")
    TARGET_FAKE_FOR_MAIN_TRAIN_VAL = 13000 

manual_test_filepaths, manual_test_labels, \
filepaths_for_main_process, labels_for_main_process = get_and_split_audio_files_with_manual_test(
    real_dir=REAL_AUDIO_PATH,
    fake_dir_parent=FAKE_AUDIO_PATH_PARENT,
    fake_ljspeech_keyword_for_main_train="ljspeech", 
    target_total_fake_for_main_train=TARGET_FAKE_FOR_MAIN_TRAIN_VAL,
    num_manual_test_samples_per_class=NUM_MANUAL_TEST_SAMPLES,
    keyword_for_manual_test_fake=None, 
    manual_test_export_path=MANUAL_TEST_EXPORT_DIR
)

if manual_test_filepaths:
    df_manual_test = pd.DataFrame({'filepath': manual_test_filepaths, 'label': manual_test_labels})
    manual_csv_path = os.path.join(MANUAL_TEST_EXPORT_DIR, 'manual_test_set_list.csv')
    df_manual_test.to_csv(manual_csv_path, index=False)
    print(f"Manual test set file list saved to: {manual_csv_path}")

if not filepaths_for_main_process:
    raise ValueError("Halting: No audio files left for the main training/validation/auto-test process.")

print(f"\nTotal samples for Main Training/Validation/Auto-Test: {len(filepaths_for_main_process)} "
      f"(Labels: Real={labels_for_main_process.count(0)}, Fake={labels_for_main_process.count(1)})")

X_train_paths, X_val_paths, X_test_auto_paths = [], [], []
y_train, y_val, y_test_auto = [], [], []

if len(filepaths_for_main_process) > 0 : 
    if len(filepaths_for_main_process) < 10: 
        print(f"Warning: Too few samples ({len(filepaths_for_main_process)}) for a full 3-way split. Adjusting...")
        if len(filepaths_for_main_process) > 1 :
            X_train_paths, X_val_paths, y_train, y_val = train_test_split(
                filepaths_for_main_process, labels_for_main_process,
                test_size=0.2, random_state=SEED, 
                stratify=labels_for_main_process if (len(set(labels_for_main_process)) > 1 and all(labels_for_main_process.count(l) >= 1 for l in set(labels_for_main_process))) else None 
            )
        else: 
            X_train_paths, y_train = list(filepaths_for_main_process), list(labels_for_main_process)
    else:
        can_stratify_main = len(set(labels_for_main_process)) > 1 and \
                            all(labels_for_main_process.count(l) >= 2 for l in set(labels_for_main_process))
        
        X_train_val_paths, X_test_auto_paths, y_train_val, y_test_auto = train_test_split(
            filepaths_for_main_process, labels_for_main_process,
            test_size=TEST_RATIO, 
            random_state=SEED,
            stratify=labels_for_main_process if can_stratify_main else None)

        if X_train_val_paths: 
             if len(X_train_val_paths) < 2:
                X_train_paths, y_train = X_train_val_paths, y_train_val
             else:
                can_stratify_train_val = len(set(y_train_val)) > 1 and \
                                         all(y_train_val.count(l) >= 2 for l in set(y_train_val))
                
                effective_val_ratio_on_train_val_set = VALIDATION_RATIO / (TRAIN_RATIO + VALIDATION_RATIO) if (TRAIN_RATIO + VALIDATION_RATIO) > 0 else 0
                if effective_val_ratio_on_train_val_set >= 1.0 or effective_val_ratio_on_train_val_set == 0 and len(X_train_val_paths) > 1 : 
                    effective_val_ratio_on_train_val_set = 0.1 if len(X_train_val_paths) > 10 else 1/len(X_train_val_paths) if len(X_train_val_paths) > 1 else 0


                X_train_paths, X_val_paths, y_train, y_val = train_test_split(
                    X_train_val_paths, y_train_val,
                    test_size=effective_val_ratio_on_train_val_set,
                    random_state=SEED,
                    stratify=y_train_val if can_stratify_train_val else None
                )
        else: 
            X_train_paths, y_train = [], []

print(f"Training samples: {len(X_train_paths)} (Real={y_train.count(0)}, Fake={y_train.count(1)})")
print(f"Validation samples: {len(X_val_paths)} (Real={y_val.count(0)}, Fake={y_val.count(1)})")
print(f"Auto-Test samples: {len(X_test_auto_paths)} (Real={y_test_auto.count(0) if isinstance(y_test_auto, list) and y_test_auto else 0}, Fake={y_test_auto.count(1) if isinstance(y_test_auto, list) and y_test_auto else 0})")

--- Starting Data Preparation: Separating Manual Test Set & Preparing Main Train/Val/Auto-Test Data ---
Selected 100 Real files for manual test.
13000 Real files remaining for main process.
Selected 100 Fake files randomly for manual test.
Exported manual test files to /kaggle/working/manual_dataset_export
Manual test set created with 200 files (Real: 100, Fake: 100).
Found 107893 LJSpeech-related fake files in remaining pool for main process.
Randomly sampled 13000 filtered fake files for main process from 107893 available.
Total files for main process (train/val/auto-test): 26000 (Real: 13000, Fake: 13000).
--- Data Preparation Finished ---
Manual test set file list saved to: /kaggle/working/manual_dataset_export/manual_test_set_list.csv

Total samples for Main Training/Validation/Auto-Test: 26000 (Labels: Real=13000, Fake=13000)
Training samples: 18199 (Real=9099, Fake=9100)
Validation samples: 3901 (Real=1951, Fake=1950)
Auto-Test samples: 3900 (Real=1950, Fake=1950)


In [42]:
# --- 4. PyTorch Datasets and DataLoaders for ViT and CNN (Create once) ---
vad_params_for_dataset = {
    'vad_aggressiveness': VAD_AGGRESSIVENESS,
    'vad_frame_duration_ms': VAD_FRAME_DURATION_MS,
    'min_speech_duration_ms': VAD_MIN_SPEECH_DURATION_MS,
    'target_audio_duration_sec': TARGET_AUDIO_DURATION_SEC_AFTER_VAD,
    'keep_only_first_speech_segment': VAD_KEEP_ONLY_FIRST_SEGMENT
}

# ViT Datasets & DataLoaders
train_dataset_vit = AudioDataset( 
    X_train_paths, y_train, 
    transform_spectrogram_fn=audio_to_melspectrogram,
    augment=APPLY_AUGMENTATION, 
    is_vit_input=True,
    use_vad_in_dataset=True,
    vad_params=vad_params_for_dataset
)
val_dataset_vit = AudioDataset(
    X_val_paths, y_val, 
    transform_spectrogram_fn=audio_to_melspectrogram,
    augment=False, 
    is_vit_input=True,
    use_vad_in_dataset=True,
    vad_params=vad_params_for_dataset 
)

def collate_fn_skip_none_vit(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch:
        # Return empty tensors with correct shape if batch is empty after filtering
        return torch.empty((0, 3, N_MELS, MAX_FRAMES_SPEC)), torch.empty((0,))
    return torch.utils.data.dataloader.default_collate(batch)

def collate_fn_skip_none_cnn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch:
        # Return empty tensors with correct shape
        return torch.empty((0, 1, N_MELS, MAX_FRAMES_SPEC)), torch.empty((0,))
    return torch.utils.data.dataloader.default_collate(batch)
    
train_loader_vit = DataLoader(train_dataset_vit, BATCH_SIZE, True, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_vit)
val_loader_vit = DataLoader(val_dataset_vit, BATCH_SIZE, False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_vit)

test_loader_vit = None 
if X_test_auto_paths and isinstance(y_test_auto, list) and y_test_auto: 
    test_dataset_vit_auto = AudioDataset(
        X_test_auto_paths, y_test_auto, 
        transform_spectrogram_fn=audio_to_melspectrogram, 
        augment=False, is_vit_input=True,
        use_vad_in_dataset=True, vad_params=vad_params_for_dataset
    )
    test_loader_vit = DataLoader(test_dataset_vit_auto, BATCH_SIZE, False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_vit)
else:
    print("Auto-Test set for ViT is empty or y_test_auto is invalid, test_loader_vit (for auto eval) not created.")


# CNN Datasets & DataLoaders
train_dataset_cnn = AudioDataset(
    X_train_paths, y_train, 
    transform_spectrogram_fn=audio_to_melspectrogram,
    augment=APPLY_AUGMENTATION, 
    is_vit_input=False,
    use_vad_in_dataset=True, 
    vad_params=vad_params_for_dataset
)
val_dataset_cnn = AudioDataset(
    X_val_paths, y_val, 
    transform_spectrogram_fn=audio_to_melspectrogram,
    augment=False, 
    is_vit_input=False,
    use_vad_in_dataset=True, 
    vad_params=vad_params_for_dataset
)
train_loader_cnn = DataLoader(train_dataset_cnn, BATCH_SIZE, True, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_cnn)
val_loader_cnn = DataLoader(val_dataset_cnn, BATCH_SIZE, False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_cnn)

test_loader_cnn = None
if X_test_auto_paths and isinstance(y_test_auto, list) and y_test_auto:
    test_dataset_cnn_auto = AudioDataset(
        X_test_auto_paths, y_test_auto, 
        transform_spectrogram_fn=audio_to_melspectrogram, 
        augment=False, is_vit_input=False,
        use_vad_in_dataset=True, vad_params=vad_params_for_dataset
    )
    test_loader_cnn = DataLoader(test_dataset_cnn_auto, BATCH_SIZE, False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_cnn)
else:
    print("Auto-Test set for CNN is empty or y_test_auto is invalid, test_loader_cnn (for auto eval) not created.")

In [43]:
# --- 5. PyTorch Vision Transformer (ViT) Model ---
# (Dán lại code model ViT ở đây)
class PatchEmbed(nn.Module):
    def __init__(self, img_size=(N_MELS, MAX_FRAMES_SPEC), patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size, img_size[1] // patch_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x) 
        x = self.fc2(x)
        x = self.drop(x) 
        return x

class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, img_size=(N_MELS, MAX_FRAMES_SPEC), patch_size=16, in_chans=3, num_classes=1,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4., qkv_bias=True,
                 drop_rate=0., attn_drop_rate=0.): 
        super().__init__()
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim)) 
        self.pos_drop = nn.Dropout(p=drop_rate)

        self.blocks = nn.ModuleList([
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias,
                  drop=drop_rate, attn_drop=attn_drop_rate)
            for _ in range(depth)]) 
        self.norm = nn.LayerNorm(embed_dim) 
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=.02)
        nn.init.trunc_normal_(self.cls_token, std=.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None: 
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_features(self, x):
        B = x.shape[0]
        x = self.patch_embed(x) 
        cls_tokens = self.cls_token.expand(B, -1, -1)  
        x = torch.cat((cls_tokens, x), dim=1) 
        x = x + self.pos_embed
        x = self.pos_drop(x) 
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x[:, 0] 

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x) 
        return x

In [44]:
# --- 6. PyTorch CNN Model (Modified for Parameterization) ---
class AudioCNN(nn.Module):
    def __init__(self, num_classes=1, dropout_rate=0.4,
                 channels_list=None, fc_nodes_list=None,
                 n_mels=N_MELS, max_frames_spec=MAX_FRAMES_SPEC):
        super(AudioCNN, self).__init__()

        if channels_list is None: 
            channels_list = [32, 64, 128, 256]
        if fc_nodes_list is None: 
            fc_nodes_list = [512, 128]

        self.conv_layers = nn.ModuleList()
        self.bn_conv_layers = nn.ModuleList()
        self.pool_layers = nn.ModuleList()
        self.drop_conv_layers = nn.ModuleList()

        in_channels = 1 
        current_height = n_mels
        current_width = max_frames_spec
        for i, out_channels in enumerate(channels_list):
            self.conv_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
            self.bn_conv_layers.append(nn.BatchNorm2d(out_channels))
            self.pool_layers.append(nn.MaxPool2d(kernel_size=2)) 
            current_dropout_rate = dropout_rate / 2 if i < len(channels_list) / 2 else dropout_rate
            self.drop_conv_layers.append(nn.Dropout2d(current_dropout_rate))
            in_channels = out_channels
            current_height = current_height // 2 
            current_width = current_width // 2  
        
        fc_in_features = channels_list[-1] * current_height * current_width

        self.fc_layers = nn.ModuleList()
        self.bn_fc_layers = nn.ModuleList()
        self.drop_fc_layers = nn.ModuleList()

        current_fc_in_dim = fc_in_features
        for i, fc_out_dim in enumerate(fc_nodes_list):
            self.fc_layers.append(nn.Linear(current_fc_in_dim, fc_out_dim))
            self.bn_fc_layers.append(nn.BatchNorm1d(fc_out_dim))
            self.drop_fc_layers.append(nn.Dropout(dropout_rate)) 
            current_fc_in_dim = fc_out_dim
        
        self.output_fc = nn.Linear(current_fc_in_dim, num_classes)

    def forward(self, x):
        for i in range(len(self.conv_layers)):
            x = self.conv_layers[i](x)
            x = self.bn_conv_layers[i](x)
            x = F.relu(x)
            x = self.pool_layers[i](x)
            x = self.drop_conv_layers[i](x)
        
        x = x.view(x.size(0), -1)
        
        for i in range(len(self.fc_layers)):
            x = self.fc_layers[i](x)
            x = self.bn_fc_layers[i](x)
            x = F.relu(x)
            x = self.drop_fc_layers[i](x)
            
        x = self.output_fc(x)
        return x

In [45]:
# --- 7. Training Loop and Evaluation Function ---
def train_one_epoch(model, train_loader, criterion, optimizer, device, epoch_num, num_epochs, model_name="Model"):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch_num+1}/{num_epochs} [{model_name} Training]", unit="batch", leave=False)
    for inputs, labels in progress_bar:
        if inputs.nelement() == 0: continue 
        inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = torch.sigmoid(outputs) > 0.5
        correct_predictions += (preds == labels).sum().item()
        total_samples += labels.size(0)
        
        if total_samples > 0:
            progress_bar.set_postfix(loss=loss.item(), acc=correct_predictions/total_samples)
        else:
            progress_bar.set_postfix(loss=loss.item(), acc=0)
            
    epoch_loss = running_loss / total_samples if total_samples > 0 else 0
    epoch_acc = correct_predictions / total_samples if total_samples > 0 else 0
    return epoch_loss, epoch_acc

def evaluate_model_pytorch(model, val_loader, criterion, device, epoch_num=None, num_epochs=None, model_name="Model", is_test_set=False):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_labels_true = []
    all_preds_probs = []
    
    desc_str = f"{model_name} Evaluating"
    if not is_test_set and epoch_num is not None and num_epochs is not None:
        desc_str = f"Epoch {epoch_num+1}/{num_epochs} [{model_name} Validation]"
    elif is_test_set:
        desc_str = f"[{model_name} Test Set Evaluation]"

    progress_bar = tqdm(val_loader, desc=desc_str, unit="batch", leave=False)
    with torch.no_grad():
        for inputs, labels in progress_bar:
            if inputs.nelement() == 0: continue
            inputs, labels_true_batch = inputs.to(device), labels.to(device).unsqueeze(1)
            
            outputs = model(inputs)
            if criterion: 
                loss_val = criterion(outputs, labels_true_batch) # Đổi tên biến loss để không ghi đè biến global
                running_loss += loss_val.item() * inputs.size(0)
            else:
                loss_val = None 
            
            probs = torch.sigmoid(outputs)
            preds_binary = probs > 0.5
            
            correct_predictions += (preds_binary == labels_true_batch).sum().item()
            total_samples += labels_true_batch.size(0)
            
            all_labels_true.extend(labels_true_batch.cpu().numpy().flatten())
            all_preds_probs.extend(probs.cpu().numpy().flatten())

            current_acc = correct_predictions/total_samples if total_samples > 0 else 0
            if loss_val is not None:
                progress_bar.set_postfix(loss=loss_val.item(), acc=current_acc)
            else:
                progress_bar.set_postfix(acc=current_acc)

    epoch_loss = running_loss / total_samples if total_samples > 0 and criterion else float('nan') 
    epoch_acc = correct_predictions / total_samples if total_samples > 0 else 0
    
    return epoch_loss, epoch_acc, np.array(all_labels_true), np.array(all_preds_probs)

def plot_history(train_losses, val_losses, train_accs, val_accs, model_name_display, filename_base):
    epochs_range = range(1, len(train_losses) + 1)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, train_losses, label='Training Loss')
    plt.plot(epochs_range, val_losses, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title(f'{model_name_display} - Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, train_accs, label='Training Accuracy')
    plt.plot(epochs_range, val_accs, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title(f'{model_name_display} - Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')

    plt.tight_layout()
    save_path = f"training_history_{filename_base}.png"
    plt.savefig(save_path)
    wandb.log({f"training_history_plot": wandb.Image(save_path)}) 
    plt.close()

In [ ]:
# --- 8. Model Configurations ---
model_configurations = [
    {"model_type": "ViT", "size": "Small", "params": {"embed_dim": 192, "depth": 5, "num_heads": 6, "mlp_ratio": 4.0}, "is_vit_input": True},
    {"model_type": "ViT", "size": "Medium", "params": {"embed_dim": 384, "depth": 6, "num_heads": 6, "mlp_ratio": 4.0}, "is_vit_input": True},
    {"model_type": "ViT", "size": "Large", "params": {"embed_dim": 512, "depth": 6, "num_heads": 8, "mlp_ratio": 4.0}, "is_vit_input": True},
    {"model_type": "CNN", "size": "Small", "params": {"channels": [16, 32, 64, 128], "fc_nodes": [128, 32]}, "is_vit_input": False},
    {"model_type": "CNN", "size": "Medium", "params": {"channels": [32, 64, 128, 256], "fc_nodes": [256, 128]}, "is_vit_input": False},
    {"model_type": "CNN", "size": "Large", "params": {"channels": [32, 64, 128, 256], "fc_nodes": [512, 128]}, "is_vit_input": False},
]

# --- 9. Main Training and Evaluation Loop ---
common_criterion = nn.BCEWithLogitsLoss()

if N_MELS % VIT_PATCH_SIZE != 0 or MAX_FRAMES_SPEC % VIT_PATCH_SIZE != 0:
    eff_H_vit = (N_MELS // VIT_PATCH_SIZE) * VIT_PATCH_SIZE
    eff_W_vit = (MAX_FRAMES_SPEC // VIT_PATCH_SIZE) * VIT_PATCH_SIZE
    print(f"Global Warning: For ViT models with patch_size={VIT_PATCH_SIZE}:")
    print(f"  N_MELS ({N_MELS}) or MAX_FRAMES_SPEC ({MAX_FRAMES_SPEC}) may not be perfectly divisible.")
    print(f"  Effective input to PatchEmbed will be ({eff_H_vit}, {eff_W_vit}).")


for config_idx, config in enumerate(model_configurations):
    model_type = config["model_type"]
    model_size = config["size"]
    model_specific_params = config["params"]
    
    run_name = f"{time.strftime('%Y%m%d_%H%M%S')}_{model_type}_{model_size}"
    wandb.init(
        project="ASM01_DAT301m_MultiModel_VAD",
        name=run_name,
        config={
            "learning_rate": LEARNING_RATE, "epochs_max": EPOCHS, "batch_size": BATCH_SIZE,
            "weight_decay": WEIGHT_DECAY, "seed": SEED, "optimizer": "AdamW",
            "sr": SR, "n_fft": N_FFT, "hop_length": HOP_LENGTH, "n_mels": N_MELS,
            "max_frames_spec": MAX_FRAMES_SPEC, "apply_augmentation": APPLY_AUGMENTATION,
            "vad_aggressiveness": VAD_AGGRESSIVENESS, # Log VAD params
            "vad_frame_duration_ms": VAD_FRAME_DURATION_MS,
            "vad_min_speech_duration_ms": VAD_MIN_SPEECH_DURATION_MS,
            "target_audio_duration_sec_after_vad": TARGET_AUDIO_DURATION_SEC_AFTER_VAD,
            "vad_keep_only_first_segment": VAD_KEEP_ONLY_FIRST_SEGMENT,
            "train_samples_count": len(X_train_paths),
            "val_samples_count": len(X_val_paths),
            "auto_test_samples_count": len(X_test_auto_paths) if X_test_auto_paths and y_test_auto else 0,
            "manual_test_samples_total": len(manual_test_filepaths) if 'manual_test_filepaths' in globals() and manual_test_filepaths else 0,
            "model_type": model_type, "model_size": model_size,
            **model_specific_params
        }
    )

    if model_type == "ViT":
        wandb.config.update({
            "vit_patch_size": VIT_PATCH_SIZE, "vit_drop_rate": VIT_BASE_DROP_RATE,
            "vit_attn_drop_rate": VIT_BASE_ATTN_DROP_RATE,
        })
    elif model_type == "CNN":
        wandb.config.update({"cnn_dropout_rate": CNN_BASE_DROPOUT_RATE})

    print(f"\n--- [{config_idx+1}/{len(model_configurations)}] Processing: {model_type} ({model_size}) ---")

    if model_type == "ViT":
        model = VisionTransformer(
            img_size=(N_MELS, MAX_FRAMES_SPEC), patch_size=VIT_PATCH_SIZE, in_chans=3, num_classes=1,
            embed_dim=model_specific_params["embed_dim"], depth=model_specific_params["depth"],
            num_heads=model_specific_params["num_heads"], mlp_ratio=model_specific_params["mlp_ratio"],
            qkv_bias=True, drop_rate=VIT_BASE_DROP_RATE, attn_drop_rate=VIT_BASE_ATTN_DROP_RATE
        ).to(DEVICE)
        current_train_loader = train_loader_vit
        current_val_loader = val_loader_vit
        current_test_loader = test_loader_vit 
    else: 
        model = AudioCNN(
            num_classes=1, dropout_rate=CNN_BASE_DROPOUT_RATE,
            channels_list=model_specific_params["channels"], fc_nodes_list=model_specific_params["fc_nodes"],
            n_mels=N_MELS, max_frames_spec=MAX_FRAMES_SPEC
        ).to(DEVICE)
        current_train_loader = train_loader_cnn
        current_val_loader = val_loader_cnn
        current_test_loader = test_loader_cnn 

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model: {model_type} ({model_size}), Trainable Parameters: {total_params:,}")
    wandb.log({"total_trainable_parameters": total_params})
    
    sample_loader_for_img = current_train_loader
    if sample_loader_for_img and len(sample_loader_for_img) > 0:
        try:
            sample_batch_x, sample_batch_y = next(iter(sample_loader_for_img))
            if sample_batch_x.nelement() > 0:
                plt.figure(figsize=(10, 4))
                img_to_show = sample_batch_x[0, 0, :, :].cpu().numpy()
                librosa.display.specshow(img_to_show, sr=SR, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel')
                plt.colorbar(format='%+2.0f dB')
                plt.title(f'Sample Input Spectrogram ({model_type} {model_size}, Label: {sample_batch_y[0].item():.0f})')
                plt.tight_layout()
                img_path = f"sample_spectrogram_{model_type.lower()}_{model_size.lower()}.png"
                plt.savefig(img_path)
                wandb.log({"sample_input_spectrogram": wandb.Image(img_path)})
                plt.close()
            else: print(f"  Skipping sample spectrogram: First batch from {model_type} loader is empty.")
        except StopIteration: print(f"  Skipping sample spectrogram: {model_type} Train loader is empty.")
        except Exception as e_img: print(f"  Error logging sample spectrogram: {e_img}")
    else: print(f"  Skipping sample spectrogram: {model_type} Train loader is unavailable or empty.")

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    history_train_loss, history_val_loss = [], []
    history_train_acc, history_val_acc = [], []
    best_val_loss_for_model = float('inf')
    best_epoch_for_model = -1
    patience_counter_for_model = 0
    model_save_path = f'best_{model_type.lower()}_{model_size.lower()}_model.pth'

    print(f"  Starting training for {model_type} ({model_size}) on {DEVICE}...")
    start_time_total_train = time.time()
    for epoch in range(EPOCHS):
        epoch_start_time = time.time()
        
        train_loss, train_acc = train_one_epoch(
            model, current_train_loader, common_criterion, optimizer, DEVICE,
            epoch, EPOCHS, model_name=f"{model_type} {model_size}"
        )
        val_loss, val_acc, _, _ = evaluate_model_pytorch(
            model, current_val_loader, common_criterion, DEVICE,
            epoch, EPOCHS, model_name=f"{model_type} {model_size}"
        )
        epoch_duration = time.time() - epoch_start_time

        print(f"    Epoch {epoch+1}/{EPOCHS} - {model_type} {model_size} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} | "
              f"Duration: {epoch_duration:.2f}s")
        
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss, "train_accuracy": train_acc,
            "val_loss": val_loss, "val_accuracy": val_acc,
            "epoch_duration_seconds": epoch_duration
        })

        history_train_loss.append(train_loss)
        history_val_loss.append(val_loss)
        history_train_acc.append(train_acc)
        history_val_acc.append(val_acc)

        if val_loss < best_val_loss_for_model:
            best_val_loss_for_model = val_loss
            best_epoch_for_model = epoch + 1
            torch.save(model.state_dict(), model_save_path)
            print(f"      Epoch {epoch+1}: Val loss improved to {val_loss:.4f}. Model saved to {model_save_path}")
            patience_counter_for_model = 0
        else:
            patience_counter_for_model += 1
            print(f"      Epoch {epoch+1}: Val loss ({val_loss:.4f}) did not improve from {best_val_loss_for_model:.4f}. Patience: {patience_counter_for_model}/{PATIENCE_LIMIT}")
        
        if patience_counter_for_model >= PATIENCE_LIMIT:
            print(f"    Early stopping triggered at epoch {epoch+1} for {model_type} {model_size}.")
            wandb.log({"early_stopping_epoch": epoch + 1})
            break
    
    total_training_time_model = time.time() - start_time_total_train
    print(f"  --- {model_type} ({model_size}) Training Finished ---")
    print(f"  Total Training Time: {total_training_time_model // 60:.0f}m {total_training_time_model % 60:.0f}s")
    print(f"  Best validation loss: {best_val_loss_for_model:.4f} at epoch {best_epoch_for_model}")
    
    wandb.log({
        "total_training_time_minutes": total_training_time_model / 60,
        "best_val_loss": best_val_loss_for_model,
        "best_epoch": best_epoch_for_model,
        "completed_epochs": epoch + 1 
    })
    if os.path.exists(model_save_path):
        model_artifact = wandb.Artifact(f"{model_type.lower()}_{model_size.lower()}_model", type="model")
        model_artifact.add_file(model_save_path)
        wandb.log_artifact(model_artifact)

    plot_filename_base = f"{model_type.lower()}_{model_size.lower()}"
    plot_history(history_train_loss, history_val_loss, history_train_acc, history_val_acc,
                 model_name_display=f"{model_type} {model_size}", filename_base=plot_filename_base)

    # 5. Test Set Evaluation (TRÊN TẬP AUTO-TEST)
    if current_test_loader and os.path.exists(model_save_path): 
        print(f"\n  --- Evaluating {model_type} ({model_size}) on Auto-Test Set ---")
        
        if model_type == "ViT":
            test_model = VisionTransformer(
                img_size=(N_MELS, MAX_FRAMES_SPEC), patch_size=VIT_PATCH_SIZE, in_chans=3, num_classes=1,
                embed_dim=model_specific_params["embed_dim"], depth=model_specific_params["depth"],
                num_heads=model_specific_params["num_heads"], mlp_ratio=model_specific_params["mlp_ratio"],
                qkv_bias=True, drop_rate=VIT_BASE_DROP_RATE, attn_drop_rate=VIT_BASE_ATTN_DROP_RATE
            ).to(DEVICE)
        else: 
            test_model = AudioCNN(
                num_classes=1, dropout_rate=CNN_BASE_DROPOUT_RATE,
                channels_list=model_specific_params["channels"], fc_nodes_list=model_specific_params["fc_nodes"],
                n_mels=N_MELS, max_frames_spec=MAX_FRAMES_SPEC
            ).to(DEVICE)
        
        try:
            test_model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
            print(f"    Best {model_type} ({model_size}) model weights loaded from {model_save_path}")
            
            test_loss, test_acc, test_labels_true, test_preds_probs = evaluate_model_pytorch(
                test_model, current_test_loader, common_criterion, DEVICE,
                model_name=f"AutoTest_{model_type}_{model_size}", is_test_set=True
            )
            print(f"    {model_type} ({model_size}) Auto-Test Set - Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}")
            wandb.log({
                "auto_test_loss": test_loss, 
                "auto_test_accuracy": test_acc
            })

            if len(test_labels_true) > 0 and len(test_preds_probs) > 0:
                test_preds_binary = (test_preds_probs > 0.5).astype(int)
                report_dict = classification_report(test_labels_true, test_preds_binary, target_names=['Real (0)', 'Fake (1)'], output_dict=True, zero_division=0)
                report_str = classification_report(test_labels_true, test_preds_binary, target_names=['Real (0)', 'Fake (1)'], zero_division=0)
                print("\n    Classification Report (Auto-Test Set):")
                print(report_str)
                wandb.log({
                    "auto_test_cls_report_str": report_str,
                    "auto_test_precision_real": report_dict['Real (0)']['precision'],
                    "auto_test_recall_real": report_dict['Real (0)']['recall'],
                    "auto_test_f1_real": report_dict['Real (0)']['f1-score'],
                    "auto_test_precision_fake": report_dict['Fake (1)']['precision'],
                    "auto_test_recall_fake": report_dict['Fake (1)']['recall'],
                    "auto_test_f1_fake": report_dict['Fake (1)']['f1-score'],
                    # ... (thêm các metric khác nếu cần)
                })
                try:
                    if len(np.unique(test_labels_true)) > 1: 
                        roc_auc = roc_auc_score(test_labels_true, test_preds_probs)
                        print(f"    ROC AUC Score (Auto-Test Set): {roc_auc:.4f}")
                        wandb.log({"auto_test_roc_auc": roc_auc})
                    else:
                        print("    ROC AUC Score (Auto-Test Set): Not calculated (only one class present).")
                        wandb.log({"auto_test_roc_auc": float('nan')}) 
                except ValueError as e_roc:
                     print(f"    Could not calculate ROC AUC for Auto-Test: {e_roc}")
                     wandb.log({"auto_test_roc_auc": float('nan')})

                cm = confusion_matrix(test_labels_true, test_preds_binary)
                disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Real', 'Fake'])
                disp.plot(cmap=plt.cm.Blues)
                plt.title(f'CM - Auto-Test - {model_type} {model_size}')
                cm_filename = f"cm_auto_test_{model_type.lower()}_{model_size.lower()}.png"
                plt.savefig(cm_filename)
                plt.close()
                wandb.log({"auto_test_confusion_matrix": wandb.Image(cm_filename)})
            else:
                print("    Not enough data in auto-test results for report/matrix.")
        except FileNotFoundError:
            print(f"    Error: Model file '{model_save_path}' not found for auto-test evaluation.")
        except Exception as e_test:
            print(f"    An error occurred during {model_type} ({model_size}) auto-test set evaluation: {e_test}")
    elif not current_test_loader:
        print(f"\n  {model_type} ({model_size}) Auto-Test loader is not available. Skipping auto-test set evaluation.")
    # ... (các else if khác cho os.path.exists(model_save_path) )

    wandb.finish()

print("\n--- All model configurations processed ---")

Global Warning: For ViT models with patch_size=16:
  N_MELS (128) or MAX_FRAMES_SPEC (313) may not be perfectly divisible.
  Effective input to PatchEmbed will be (128, 304).



--- [1/6] Processing: ViT (Small) ---
  Model: ViT (Small), Trainable Parameters: 2,402,113
  Starting training for ViT (Small) on cuda...


Epoch 1/20 [ViT Small Training]:   0%|          | 0/569 [00:00<?, ?batch/s]

In [ ]:
# --- TÙY CHỌN: ĐÁNH GIÁ CÁC MODEL TỐT NHẤT TRÊN TẬP MANUAL TEST ---
if 'manual_test_filepaths' in globals() and 'manual_test_labels' in globals() and manual_test_filepaths:
    print("\n\n--- OPTIONAL: Evaluating Best Models on Manual Test Set ---")
    
    # Tạo DataLoader cho manual test (chỉ cần làm một lần cho mỗi loại input)
    manual_test_dataset_vit = AudioDataset(manual_test_filepaths, manual_test_labels, audio_to_melspectrogram, False, True, use_vad_in_dataset=True, vad_params=vad_params_for_dataset)
    manual_test_loader_vit = DataLoader(manual_test_dataset_vit, BATCH_SIZE, False, num_workers=NUM_WORKERS, collate_fn=collate_fn_skip_none_vit)

    manual_test_dataset_cnn = AudioDataset(manual_test_filepaths, manual_test_labels, audio_to_melspectrogram, False, False, use_vad_in_dataset=True, vad_params=vad_params_for_dataset)
    manual_test_loader_cnn = DataLoader(manual_test_dataset_cnn, BATCH_SIZE, False, num_workers=NUM_WORKERS, collate_fn=collate_fn_skip_none_cnn)

    for config_idx, config in enumerate(model_configurations):
        model_type = config["model_type"]
        model_size = config["size"]
        model_specific_params = config["params"]
        saved_model_path = f'best_{model_type.lower()}_{model_size.lower()}_model.pth' 
        
        if os.path.exists(saved_model_path):
            print(f"\n--- Evaluating {model_type} {model_size} on MANUAL Test Set ---")
            
            if model_type == "ViT":
                eval_manual_model = VisionTransformer(
                    img_size=(N_MELS, MAX_FRAMES_SPEC), patch_size=VIT_PATCH_SIZE, in_chans=3, num_classes=1,
                    embed_dim=model_specific_params["embed_dim"], depth=model_specific_params["depth"],
                    num_heads=model_specific_params["num_heads"], mlp_ratio=model_specific_params["mlp_ratio"],
                    qkv_bias=True, drop_rate=VIT_BASE_DROP_RATE, attn_drop_rate=VIT_BASE_ATTN_DROP_RATE
                ).to(DEVICE)
                current_manual_loader = manual_test_loader_vit
            else: 
                eval_manual_model = AudioCNN(
                    num_classes=1, dropout_rate=CNN_BASE_DROPOUT_RATE,
                    channels_list=model_specific_params["channels"], fc_nodes_list=model_specific_params["fc_nodes"],
                    n_mels=N_MELS, max_frames_spec=MAX_FRAMES_SPEC
                ).to(DEVICE)
                current_manual_loader = manual_test_loader_cnn
            
            try:
                eval_manual_model.load_state_dict(torch.load(saved_model_path, map_location=DEVICE))
                print(f"  Successfully loaded: {saved_model_path}")

                manual_loss, manual_acc, manual_true, manual_probs = evaluate_model_pytorch(
                    eval_manual_model, current_manual_loader, common_criterion, DEVICE, 
                    model_name=f"ManualTest_{model_type}_{model_size}", is_test_set=True
                )
                print(f"  MANUAL Test - {model_type} {model_size} - Loss: {manual_loss:.4f}, Accuracy: {manual_acc:.4f}")
                
                if len(manual_true) > 0:
                    manual_preds_binary = (manual_probs > 0.5).astype(int)
                    print("\n  Classification Report (Manual Test Set):")
                    print(classification_report(manual_true, manual_preds_binary, target_names=['Real (0)', 'Fake (1)'], zero_division=0))
                    
                    cm_manual = confusion_matrix(manual_true, manual_preds_binary)
                    disp_manual = ConfusionMatrixDisplay(confusion_matrix=cm_manual, display_labels=['Real', 'Fake'])
                    plt.figure()
                    disp_manual.plot(cmap=plt.cm.Blues)
                    plt.title(f'CM - MANUAL Test - {model_type} {model_size}')
                    cm_manual_filename = f"cm_manual_test_{model_type.lower()}_{model_size.lower()}.png"
                    plt.savefig(cm_manual_filename)
                    print(f"  Saved manual test confusion matrix to {cm_manual_filename}")
                    plt.close() 
            except Exception as e_manual_eval:
                print(f"  Error during manual test evaluation for {saved_model_path}: {e_manual_eval}")
        else:
            print(f"Model file {saved_model_path} not found. Skipping manual test evaluation for this model.")
else:
    print("Manual test file list ('manual_test_filepaths') not found. Skipping manual test evaluation section.")